# NLP Project

ANTIPIN Vladislav, OUNDJIAN Milos

## Imports

In [9]:
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path
import string
# from rital.preprocessing import load_presidents, load_movies

In [10]:
from rital.data import (
    load_presidents,
    load_presidents_unseen,
    load_movies,
    load_movies_unseen,
)
from rital.preprocessing import preprocess

## Data Loading

In [11]:
x_presidents, y_presidents = load_presidents()
x_presidents_unseen = load_presidents_unseen()
x_movies, y_movies = load_movies()
x_movies_unseen = load_movies_unseen()

## Basic TF-IDF Classifier

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression  # TODO we can also use this
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import classification_report

# Data
# Switch this between presidents and movies
x = x_movies
y = y_movies

# TF-IDF + SVM pipeline
model = Pipeline(
    [
        ("tfidf", TfidfVectorizer(preprocessor=preprocess)),
        ("classifier", LinearSVC()),
    ]
)

cv = StratifiedKFold(5, shuffle=True, random_state=42)

scores = cross_validate(
    model, x, y, cv=cv, scoring=["accuracy", "f1", "precision", "recall"]
)

for metric, values in scores.items():
    if "test" in metric:
        print(f"{metric}: {values.mean():.3f}")


y_pred = cross_val_predict(model, x, y, cv=cv)

print(classification_report(y, y_pred))

test_accuracy: 0.858
test_f1: 0.860
test_precision: 0.847
test_recall: 0.873
              precision    recall  f1-score   support

          -1       0.87      0.84      0.86      1000
           1       0.85      0.87      0.86      1000

    accuracy                           0.86      2000
   macro avg       0.86      0.86      0.86      2000
weighted avg       0.86      0.86      0.86      2000



## Hyperparameter Tuning

~ 1m runtime

In [47]:
from sklearn.model_selection import GridSearchCV

# Data
x = x_movies
y = y_movies

# CV splitter
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Pipeline
model = Pipeline(
    [
        ("tfidf", TfidfVectorizer(preprocessor=preprocess)),
        ("classifier", LinearSVC()),
    ]
)

# Hyperparameter grid
param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2, 5],
    "tfidf__max_df": [0.9, 1.0],
    "tfidf__sublinear_tf": [False, True],
    "classifier__C": [0.01, 0.1, 1, 10],
}

# Grid search
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    scoring="f1",  # or "accuracy"
    n_jobs=-1,
    verbose=1,
)

grid.fit(x, y)

print(f"Best parameters: {grid.best_params_}")
print(f"Best CV score: {grid.best_score_}")

best_model = grid.best_estimator_

y_pred = cross_val_predict(best_model, x, y, cv=cv)

print("Classification report:")
print(classification_report(y, y_pred))

Fitting 5 folds for each of 96 candidates, totalling 480 fits
Best parameters: {'classifier__C': 1, 'tfidf__max_df': 0.9, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True}
Best CV score: 0.8897131878283083
Classification report:
              precision    recall  f1-score   support

          -1       0.89      0.89      0.89      1000
           1       0.89      0.89      0.89      1000

    accuracy                           0.89      2000
   macro avg       0.89      0.89      0.89      2000
weighted avg       0.89      0.89      0.89      2000



~ 1m40s runtime

In [48]:
from sklearn.model_selection import GridSearchCV

# Data
x = x_presidents
y = y_presidents

# CV splitter
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Pipeline
model = Pipeline(
    [
        ("tfidf", TfidfVectorizer(preprocessor=preprocess)),
        ("classifier", LinearSVC()),
    ]
)

# Hyperparameter grid
param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2, 5],
    "tfidf__max_df": [0.9, 1.0],
    "tfidf__sublinear_tf": [False, True],
    "classifier__C": [0.01, 0.1, 1, 10],
}

# Grid search
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    scoring="f1",  # or "accuracy"
    n_jobs=-1,
    verbose=1,
)

grid.fit(x, y)

print(f"Best parameters: {grid.best_params_}")
print(f"Best CV score: {grid.best_score_}")

best_model = grid.best_estimator_

y_pred = cross_val_predict(best_model, x, y, cv=cv)

print("Classification report:")
print(classification_report(y, y_pred))

Fitting 5 folds for each of 96 candidates, totalling 480 fits
Best parameters: {'classifier__C': 1, 'tfidf__max_df': 0.9, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True}
Best CV score: 0.951420598118603
Classification report:
              precision    recall  f1-score   support

          -1       0.78      0.47      0.59      7523
           1       0.93      0.98      0.95     49890

    accuracy                           0.91     57413
   macro avg       0.85      0.73      0.77     57413
weighted avg       0.91      0.91      0.90     57413

